In [0]:
%sql
DROP TABLE IF EXISTS catalog_ventas.gold.kpi_top_productos;

CREATE OR REPLACE TABLE catalog_ventas.gold.kpi_top_productos
USING DELTA
AS

WITH base AS (
  SELECT
      p.descrip,
      p.categoria,
      SUM(f.total) AS facturacion,
      SUM(f.cant)  AS unidades,
      COUNT(DISTINCT f.id_venta) AS tickets

  FROM catalog_ventas.gold.fact_ventas f

  LEFT JOIN catalog_ventas.gold.dim_producto p
    ON f.id_articulo = p.id_articulo AND p.es_actual = TRUE

  GROUP BY p.descrip, p.categoria
),

ranking AS (
  SELECT
  *,
  RANK() OVER (ORDER BY facturacion DESC) AS rnk
  FROM base
)

SELECT * FROM ranking WHERE rnk <= 10